In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
REPO = "/content/LateComers-CSE576-NLP"
if not os.path.exists(REPO):
    !git clone -q https://github.com/aaryapendharkar/LateComers-CSE576-NLP.git {REPO}
%cd {REPO}
!git config user.email "p5@colab" && git config user.name "p5"
!git fetch -q origin
!git checkout -q -f -B p5-run origin/main
!git merge -q --no-edit origin/p4-renderer-v1 origin/p5-gap2-defense
!git log --oneline -1

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y poppler-utils > /dev/null
!which pdftoppm
!pip install -q -r requirements-core.txt
!pip install -q -U transformers accelerate

In [ ]:
import torch, yaml
big = torch.cuda.get_device_properties(0).total_memory / 1e9 >= 30
local = {
    "pipelines": {"backend": "real"},
    "models": {
        "generator_fallback": None if big else "Qwen/Qwen3-VL-4B-Instruct",
        "device": "cuda",
        "dtype": "bfloat16" if torch.cuda.is_bf16_supported() else "float16",
    },
    "img_text_check": {"text_source": "pdfplumber", "max_new_tokens": 1024},
}
with open("configs/local.yaml", "w") as f:
    yaml.safe_dump(local, f)
print("Qwen/Qwen3.5-9B" if big else "Qwen/Qwen3-VL-4B-Instruct")

In [ ]:
!python scripts/render_clean_pages.py --workers 2 2>&1 | tail -1
!python scripts/render_clean_pages.py --source chartqa --workers 2 2>&1 | tail -1
!python scripts/make_g2_propagation.py 2>&1 | grep -v "it/s"
!python scripts/make_g2_attacks.py 2>&1 | grep -v "it/s"
!python scripts/build_indexes.py --parse pdfplumber 2>&1 | tail -1

In [ ]:
!python scripts/eval_img_text_check.py --oracle --overwrite 2>&1 | grep -v 'it/s'

In [ ]:
!python scripts/eval_img_text_check.py --n-clean 25 --limit 5 --read-accuracy 2>&1 | grep -v 'it/s'

In [ ]:
import json
for r in map(json.loads, open("results/gap2/checker_reads.jsonl")):
    print(r["page_id"], r["exact"], r["misreads"][:3])

In [ ]:
!python scripts/eval_img_text_check.py --n-clean 25

In [ ]:
!python scripts/eval_img_text_check.py --n-clean 25 --overwrite --cached-only --read-accuracy 2>&1 | grep -v 'it/s'
print(open("results/gap2/checker_table.md").read())
print(open("results/gap2/checker_reads.md").read())

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/cse576_p5
!cp -r results/gap2 /content/drive/MyDrive/cse576_p5/
!cp -r data/parsed/img_tables /content/drive/MyDrive/cse576_p5/
!ls /content/drive/MyDrive/cse576_p5/gap2

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p data/parsed results
!cp -r /content/drive/MyDrive/cse576_p5/img_tables data/parsed/
!cp -r /content/drive/MyDrive/cse576_p5/gap2 results/